# Phase 2 — Reproducible split and preprocessing

This notebook creates stratified train/validation/test split manifests and TensorFlow datasets. It does **not** train a model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('/content/drive/MyDrive/waste-classification')
AUDIT_DIR = PROJECT_ROOT / 'outputs' / 'phase1_audit'
SPLIT_DIR = PROJECT_ROOT / 'outputs' / 'phase2_splits'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ('plastic', 'paper', 'glass', 'metal', 'organic')
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(TARGET_CLASSES)}
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_SEED = 42

manifest_path = AUDIT_DIR / 'image_manifest.csv'
assert manifest_path.exists(), 'Run the Phase 1 audit notebook first.'
manifest = pd.read_csv(manifest_path)

print(f'Images in audit manifest: {len(manifest):,}')
display(manifest.head())

In [ ]:
# 70% train, 15% validation, 15% test; stratification preserves class proportions.
train_df, temporary_df = train_test_split(
    manifest,
    test_size=0.30,
    stratify=manifest['label'],
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df['label'],
    random_state=RANDOM_SEED,
)

splits = {'train': train_df, 'validation': val_df, 'test': test_df}
for split_name, split_df in splits.items():
    split_df = split_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    split_df.to_csv(SPLIT_DIR / f'{split_name}.csv', index=False)
    splits[split_name] = split_df
    print(f'{split_name.title():10}: {len(split_df):,} images')

assert len(train_df) + len(val_df) + len(test_df) == len(manifest)
assert set(train_df['path']).isdisjoint(val_df['path'])
assert set(train_df['path']).isdisjoint(test_df['path'])
assert set(val_df['path']).isdisjoint(test_df['path'])

In [ ]:
# Show that every split contains every class in approximately the same proportion.
distribution = pd.concat(
    [
        split_df['label'].value_counts().reindex(TARGET_CLASSES, fill_value=0)
        .rename(split_name)
        for split_name, split_df in splits.items()
    ],
    axis=1,
)
display(distribution)

plot_data = distribution.reset_index(names='class').melt(
    id_vars='class', var_name='split', value_name='images'
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x='class', y='images', hue='split')
plt.title('Stratified image distribution across data splits')
plt.xlabel('Waste class')
plt.ylabel('Image count')
plt.tight_layout()
plt.savefig(SPLIT_DIR / 'split_distribution.png', dpi=160)
plt.show()

In [ ]:
# Compute class weights from the training split only. They will reduce bias
# toward the large glass class during model fitting.
train_counts = train_df['label'].value_counts().reindex(TARGET_CLASSES)
class_weights = {
    CLASS_TO_INDEX[class_name]: len(train_df) / (len(TARGET_CLASSES) * count)
    for class_name, count in train_counts.items()
}

print('Class-to-index mapping:', CLASS_TO_INDEX)
print('Class weights:', class_weights)

with open(SPLIT_DIR / 'configuration.json', 'w') as file:
    json.dump(
        {
            'target_classes': TARGET_CLASSES,
            'class_to_index': CLASS_TO_INDEX,
            'image_size': IMAGE_SIZE,
            'batch_size': BATCH_SIZE,
            'random_seed': RANDOM_SEED,
            'class_weights': class_weights,
        },
        file,
        indent=2,
    )

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

def load_and_resize_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(dataframe, training=False):
    paths = dataframe['path'].astype(str).to_numpy()
    labels = dataframe['label'].map(CLASS_TO_INDEX).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True,
        )

    return (
        dataset
        .map(load_and_resize_image, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

print('TensorFlow datasets are ready.')
print('One training batch:', next(iter(train_ds))[0].shape)

In [ ]:
# Visual verification after loading/resizing. No augmentation is applied here;
# augmentation belongs only in the training model used later.
images, labels = next(iter(train_ds.take(1)))
plt.figure(figsize=(12, 8))
for index in range(min(12, len(images))):
    axis = plt.subplot(3, 4, index + 1)
    axis.imshow(tf.cast(images[index], tf.uint8))
    axis.set_title(TARGET_CLASSES[int(labels[index])].title())
    axis.axis('off')
plt.suptitle('Preprocessed training images (224 × 224)')
plt.tight_layout()
plt.show()